# Stroke Prediction using Logistic Regression
**Oshwal College | BTEC L3 IT | Introduction to AI | Tasks 2 and 3**

Dataset: Kaggle *Stroke Prediction Dataset* (`healthcare-dataset-stroke-data.csv`).

**How to use:** Run the cells from top to bottom (Runtime > Run all, or Shift+Enter on each cell). Above each important cell there is a **📸 SCREENSHOT** note telling you what to capture and which part of the report it belongs to. Download the CSV from Kaggle first and keep it ready to upload in Step 2.

## Step 1: Import libraries (Task 2, P4 / Task 3, P5)

### 📸 SCREENSHOT 1
Run the cell below, then screenshot the code **and** the printed versions.
**Report location:** Section 2.2, under *Source*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Libraries imported successfully")

## Step 2: Load the dataset (P4)

### 📸 SCREENSHOT 2
After the file uploads, screenshot the output showing the shape and the first 5 rows.
**Report location:** Section 2.2, under *Source*.

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose healthcare-dataset-stroke-data.csv

df = pd.read_csv(list(uploaded.keys())[0])
print("Shape (rows, columns):", df.shape)
df.head()

### 📸 SCREENSHOT 3
Screenshot the `df.info()` output (data types and non-null counts; bmi has fewer entries).
**Report location:** Section 2.2, right after Screenshot 2.

In [ ]:
df.info()

## Step 3: Inspect data quality BEFORE cleaning (P4 and M2 'before')

### 📸 SCREENSHOT 4
Screenshot the missing values and duplicates output.
**Report location:** Section 2.2, under *Initial inspection*.

In [ ]:
print("Missing values per column (BEFORE):")
print(df.isnull().sum())

# Save 'before' numbers for the M2 comparison later
bmi_missing_before = int(df['bmi'].isnull().sum())
dups_before = int(df.duplicated().sum())
gender_other_before = int((df['gender'] == 'Other').sum())
stroke_counts_before = df['stroke'].value_counts()
stroke_pct_before = df['stroke'].value_counts(normalize=True) * 100

print("\nDuplicate rows:", dups_before)
print("Rows with gender = 'Other':", gender_other_before)

### 📸 SCREENSHOT 5
Screenshot the class counts, percentages and the bar chart (shows the imbalance).
**Report location:** Section 2.2, under *Initial inspection* (second screenshot there).

In [ ]:
print("Stroke class counts (BEFORE):")
print(stroke_counts_before)
print("\nPercentages:")
print(stroke_pct_before.round(2))

ax = stroke_counts_before.plot(kind='bar', color=['steelblue', 'tomato'])
plt.title("Class distribution of 'stroke' (before refinement)")
plt.xlabel("stroke (0 = no, 1 = yes)")
plt.ylabel("Number of patients")
plt.xticks(rotation=0)
for i, v in enumerate(stroke_counts_before):
    ax.text(i, v + 40, str(v), ha='center')
plt.show()

## Step 4: Clean the data (P4)

### 📸 SCREENSHOT 6
Screenshot the cleaning code and the output showing 0 missing values and the new shape.
**Report location:** Section 2.2, under *Cleaning*.

In [ ]:
df_clean = df.copy()

# 1. Drop the id column (no predictive value)
df_clean = df_clean.drop(columns=['id'])

# 2. Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# 3. Remove the single 'Other' gender record (too rare to learn from)
df_clean = df_clean[df_clean['gender'] != 'Other']

# 4. Fill missing bmi with the MEDIAN (bmi is right-skewed, median is robust)
median_bmi = df_clean['bmi'].median()
df_clean['bmi'] = df_clean['bmi'].fillna(median_bmi)

print("Median bmi used for filling:", round(median_bmi, 2))
print("Missing values AFTER cleaning:")
print(df_clean.isnull().sum())
print("\nNew shape:", df_clean.shape)

## Step 5: Encode categorical values (P4)

### 📸 SCREENSHOT 7
Screenshot `head()` showing all-numeric columns including the new dummy columns.
**Report location:** Section 2.2, under *Encoding categorical values*.

In [ ]:
df_enc = df_clean.copy()

# Binary columns -> 0/1
df_enc['gender'] = df_enc['gender'].map({'Male': 0, 'Female': 1})
df_enc['ever_married'] = df_enc['ever_married'].map({'No': 0, 'Yes': 1})
df_enc['Residence_type'] = df_enc['Residence_type'].map({'Rural': 0, 'Urban': 1})

# Multi-category columns -> one-hot encoding
df_enc = pd.get_dummies(df_enc, columns=['work_type', 'smoking_status'],
                        drop_first=True, dtype=int)

print("Any text columns left?", df_enc.select_dtypes(include='object').columns.tolist())
df_enc.head()

## Step 6: Split into training and testing sets (P4)

### 📸 SCREENSHOT 8
Screenshot the shapes and the stroke proportion in each set.
**Report location:** Section 2.2, under *Splitting*.

In [ ]:
X = df_enc.drop(columns=['stroke'])
y = df_enc['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("y_train:", y_train.shape, " y_test:", y_test.shape)
print("\nStroke % in training set:", round(y_train.mean() * 100, 2))
print("Stroke % in testing set: ", round(y_test.mean() * 100, 2))

## Step 7: Feature scaling (P4)

### 📸 SCREENSHOT 9
Screenshot both describe() tables (before and after scaling).
**Report location:** Section 2.2, under *Feature scaling*.

In [ ]:
num_cols = ['age', 'avg_glucose_level', 'bmi']

print("BEFORE scaling (training data):")
display(X_train[num_cols].describe().round(2))

scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])   # fit on training only
X_test_s[num_cols] = scaler.transform(X_test[num_cols])         # apply to test

print("AFTER scaling (training data):")
display(X_train_s[num_cols].describe().round(2))

## Step 8: Before and after data quality comparison (M2)

### 📸 SCREENSHOT 10
Screenshot the comparison table.
**Report location:** Section 2.3 (M2), under the before/after table.

In [ ]:
after_pct = y_train.value_counts(normalize=True) * 100

comparison = pd.DataFrame({
    "Quality check": ["Missing bmi values", "Duplicate rows", "'Other' gender rows",
                      "Rows in dataset", "Columns that are text", "Stroke % (class imbalance)"],
    "BEFORE": [bmi_missing_before, dups_before, gender_other_before,
               df.shape[0], int(df.select_dtypes(include='object').shape[1]),
               round(stroke_pct_before[1], 2)],
    "AFTER": [int(df_clean['bmi'].isnull().sum()), int(df_clean.duplicated().sum()),
              int((df_clean['gender'] == 'Other').sum()),
              df_enc.shape[0], int(df_enc.select_dtypes(include='object').shape[1]),
              f"{round(after_pct[1], 2)} (handled with class_weight)"]
})
comparison

## Step 9: Train the baseline Logistic Regression model (P5)

### 📸 SCREENSHOT 11
Screenshot the training cell with its output (metrics and classification report). This proves a working AI solution.
**Report location:** Section 3.1 (P5).

In [ ]:
results = []

def evaluate(name, model, X_te, y_te):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {"Model": name,
            "Accuracy": round(accuracy_score(y_te, pred), 3),
            "Precision (stroke)": round(precision_score(y_te, pred, zero_division=0), 3),
            "Recall (stroke)": round(recall_score(y_te, pred), 3),
            "F1 (stroke)": round(f1_score(y_te, pred, zero_division=0), 3),
            "ROC-AUC": round(roc_auc_score(y_te, proba), 3)}

baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train_s, y_train)

results.append(evaluate("Baseline", baseline, X_test_s, y_test))
print(results[-1])
print("\nClassification report (Baseline):")
print(classification_report(y_test, baseline.predict(X_test_s), zero_division=0))

### 📸 SCREENSHOT 12
Screenshot the baseline confusion matrix.
**Report location:** Section 3.1 (P5), after Screenshot 11.

In [ ]:
ConfusionMatrixDisplay.from_estimator(baseline, X_test_s, y_test, cmap='Blues')
plt.title("Confusion matrix: Baseline")
plt.show()

## Step 10: Test and refine the model (M3)

### 📸 SCREENSHOT 13
Screenshot the balanced model metrics, classification report and confusion matrix.
**Report location:** Section 3.2 (M3), under *Refinement 1*.

In [ ]:
balanced = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
balanced.fit(X_train_s, y_train)

results.append(evaluate("Balanced class weights", balanced, X_test_s, y_test))
print(results[-1])
print("\nClassification report (Balanced):")
print(classification_report(y_test, balanced.predict(X_test_s), zero_division=0))

ConfusionMatrixDisplay.from_estimator(balanced, X_test_s, y_test, cmap='Greens')
plt.title("Confusion matrix: Balanced class weights")
plt.show()

### 📸 SCREENSHOT 14
Screenshot the GridSearchCV code with the best parameters and best score.
**Report location:** Section 3.2 (M3), under *Refinement 2*.

In [ ]:
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
                    param_grid, scoring='recall', cv=cv)
grid.fit(X_train_s, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validated recall:", round(grid.best_score_, 3))

best_model = grid.best_estimator_
results.append(evaluate("Balanced + tuned (best C)", best_model, X_test_s, y_test))

### 📸 SCREENSHOT 15
Screenshot the comparison table of all three models.
**Report location:** Section 3.2 (M3), under *Comparison of results*. Copy these numbers into the report's [X.XX] spots.

In [ ]:
results_df = pd.DataFrame(results)
results_df

### 📸 SCREENSHOT 16
Screenshot the ROC curve plot.
**Report location:** Section 3.2 (M3), after Screenshot 15.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_estimator(baseline, X_test_s, y_test, name="Baseline", ax=ax)
RocCurveDisplay.from_estimator(best_model, X_test_s, y_test, name="Balanced + tuned", ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label="Random guessing")
ax.set_title("ROC curve comparison")
ax.legend()
plt.show()

## Step 11: Evaluate the solution (D2)

### 📸 SCREENSHOT 17
Screenshot the coefficient bar chart.
**Report location:** Section 3.3 (D2), under *Strengths*.

In [ ]:
coefs = pd.Series(best_model.coef_[0], index=X_train_s.columns).sort_values()

plt.figure(figsize=(8, 6))
coefs.plot(kind='barh', color=['tomato' if c > 0 else 'steelblue' for c in coefs])
plt.title("Logistic Regression coefficients (positive = higher stroke risk)")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()

print("Top 5 risk-increasing features:")
print(coefs.sort_values(ascending=False).head(5).round(3))

## Step 12: Save the cleaned data (for GitHub)

In [ ]:
df_enc.to_csv("stroke_cleaned.csv", index=False)
print("Saved stroke_cleaned.csv (download it from the Files panel on the left)")

## Step 13: Publish to GitHub (D2)

1. In Colab: **File > Download > Download .ipynb**. (Make sure all cells have been run so outputs are saved. Alternatively, File > Save a copy in GitHub.)
2. Go to github.com > **New repository**. Name it e.g. `stroke-prediction-logistic-regression`, choose **Public**, tick *Add a README*.
3. Click **Add file > Upload files** and upload: the `.ipynb` notebook, `README.md`, `requirements.txt` (both provided with this notebook) and optionally `stroke_cleaned.csv`.
4. Commit the changes, copy the repository URL and paste it into Section 3.3 of your report.

### 📸 SCREENSHOT 18
Screenshot the GitHub repository page showing the **Public** label, the notebook, README.md and requirements.txt.
**Report location:** Section 3.3 (D2), under *GitHub repository*.